# XGBoost Hyperparameter Tuning — Random Search
**Dataset:** CAN 2023/2018 balanced (500k rows)  
**Target:** Award outcome (binary)  
**Features:** 14 clean features (leakage removed, low importance dropped, correlated deduplicated)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
from xgboost import XGBClassifier
from scipy.stats import randint, uniform

RANDOM_STATE = 42
print('Imports loaded.')

## 1. Load & Preprocess

In [ ]:
df = pd.read_csv('../raw_data/export_CAN_2023_2018_balanced_500k.tsv', sep='\t', low_memory=False)
print(f'Loaded: {len(df):,} rows, {df.shape[1]} columns')

df['target'] = (df['INFO_ON_NON_AWARD'] == 'awarded').astype(int)

# 14 clean features — leakage removed, correlated deduplicated, low-importance dropped
FEATURES = [
    'B_MULTIPLE_CAE',    # 17.4% importance
    'B_EU_FUNDS',        # 11.2%
    'LOTS_NUMBER',       # 8.7% (NUMBER_AWARDS removed — leakage)
    'TOP_TYPE',          # 7.1%
    'YEAR',              # 6.0%
    'ISO_COUNTRY_CODE',  # 5.1%
    'TYPE_OF_CONTRACT',  # 4.8%
    'B_GPA',             # 4.4%
    'B_FRA_AGREEMENT',   # 4.3%
    'CRIT_PRICE_WEIGHT', # 2.5%
    'CRIT_CODE',         # 2.4%
    'B_ACCELERATED',     # 2.3%
    'CAE_TYPE',          # 2.1%
    'MAIN_ACTIVITY',     # 1.7%
]

# Nominal categoricals — XGBoost handles splits natively, no integer encoding needed
CAT_FEATURES = [
    'TOP_TYPE', 'ISO_COUNTRY_CODE', 'TYPE_OF_CONTRACT',
    'CRIT_CODE', 'CAE_TYPE', 'MAIN_ACTIVITY',
]

# Binary flags stored as 'Y'/'N' strings in the raw data — map to 0/1
BINARY_FEATURES = [
    'B_MULTIPLE_CAE', 'B_EU_FUNDS', 'B_GPA', 'B_FRA_AGREEMENT', 'B_ACCELERATED',
]

# Numeric features stored as strings in the raw data — coerce to float
NUMERIC_FEATURES = ['CRIT_PRICE_WEIGHT', 'LOTS_NUMBER', 'YEAR']

df_model = df[FEATURES + ['target']].copy()

for col in CAT_FEATURES:
    df_model[col] = df_model[col].fillna('missing').astype('category')

for col in BINARY_FEATURES:
    df_model[col] = df_model[col].map({'Y': 1, 'N': 0}).fillna(0).astype(int)

for col in NUMERIC_FEATURES:
    df_model[col] = pd.to_numeric(df_model[col], errors='coerce').fillna(0)

X = df_model[FEATURES]
y = df_model['target']

print(f'Target balance: {y.mean()*100:.1f}% awarded')
print(f'Features: {len(FEATURES)} ({len(CAT_FEATURES)} native categoricals, {len(BINARY_FEATURES)} binary, {len(NUMERIC_FEATURES)} numeric)')
print(f'\nDtypes:\n{X.dtypes}')

## 2. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Train: {len(X_train):,} rows')
print(f'Test:  {len(X_test):,} rows')

## 3. Baseline — Default XGBoost
Establish a reference point before tuning.

In [ ]:
baseline = XGBClassifier(
    n_estimators=100, max_depth=4,
    enable_categorical=True, tree_method='hist',
    random_state=RANDOM_STATE, n_jobs=-1,
)
baseline.fit(X_train, y_train)

baseline_auc = roc_auc_score(y_test, baseline.predict_proba(X_test)[:, 1])
baseline_acc = accuracy_score(y_test, baseline.predict(X_test))

print(f'Baseline AUC:      {baseline_auc:.4f}')
print(f'Baseline Accuracy: {baseline_acc*100:.1f}%')

## 4. Random Search — Hyperparameter Tuning

**Why random search?**  
Grid search over 6+ XGBoost parameters explodes in combinations. Random search samples the space efficiently and finds near-optimal results in ~20–50% of the compute, then you can narrow with a targeted grid around the best region.

**CV strategy:** Stratified 5-fold (preserves class balance across folds).  
**Metric:** ROC AUC (consistent with EDA evaluation).

In [ ]:
param_dist = {
    'n_estimators'      : randint(100, 600),
    'max_depth'         : randint(3, 10),
    'learning_rate'     : uniform(0.01, 0.29),   # range: 0.01 – 0.30
    'subsample'         : uniform(0.6, 0.4),      # range: 0.60 – 1.00
    'colsample_bytree'  : uniform(0.5, 0.5),      # range: 0.50 – 1.00
    'min_child_weight'  : randint(1, 10),
    'gamma'             : uniform(0, 0.5),
    'reg_alpha'         : uniform(0, 1.0),        # L1
    'reg_lambda'        : uniform(0.5, 2.0),      # L2
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

xgb_base = XGBClassifier(
    enable_categorical=True, tree_method='hist',
    random_state=RANDOM_STATE, n_jobs=-1, eval_metric='logloss',
)

random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=50,            # 50 random combinations — increase for more thorough search
    scoring='roc_auc',
    cv=cv,
    verbose=1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    refit=True,           # refit best model on full training set
)

print('Running 50-iteration random search with 5-fold CV...')
print('Total fits: 50 × 5 = 250')
random_search.fit(X_train, y_train)
print('Done.')

## 5. Results

In [ ]:
best_params = random_search.best_params_
best_cv_auc = random_search.best_score_

print('=== Best Parameters ===')
for k, v in sorted(best_params.items()):
    print(f'  {k:25s}: {v}')

print(f'\nBest CV AUC (5-fold): {best_cv_auc:.4f}')
print(f'Baseline AUC:         {baseline_auc:.4f}')
print(f'Gain:                 +{(best_cv_auc - baseline_auc):.4f}')

In [ ]:
# CV score distribution across all 50 iterations
cv_results = pd.DataFrame(random_search.cv_results_)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of mean CV AUC scores
axes[0].hist(cv_results['mean_test_score'], bins=20, color='steelblue', edgecolor='white')
axes[0].axvline(best_cv_auc, color='red', linestyle='--', label=f'Best: {best_cv_auc:.4f}')
axes[0].axvline(baseline_auc, color='orange', linestyle='--', label=f'Baseline: {baseline_auc:.4f}')
axes[0].set_title('CV AUC Distribution (50 iterations)', fontsize=13)
axes[0].set_xlabel('Mean CV AUC')
axes[0].legend()

# AUC vs iteration (ordered by rank)
sorted_scores = cv_results.sort_values('rank_test_score')['mean_test_score'].values
axes[1].plot(range(1, len(sorted_scores)+1), sorted_scores, color='steelblue', marker='o', markersize=3)
axes[1].axhline(baseline_auc, color='orange', linestyle='--', label=f'Baseline: {baseline_auc:.4f}')
axes[1].set_title('AUC by Rank (best → worst)', fontsize=13)
axes[1].set_xlabel('Rank')
axes[1].set_ylabel('Mean CV AUC')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Evaluate Best Model on Held-Out Test Set

In [ ]:
best_model = random_search.best_estimator_

y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

tuned_auc = roc_auc_score(y_test, y_prob)
tuned_acc = accuracy_score(y_test, y_pred)

print('=== Test Set Results ===')
print(f'Tuned AUC:      {tuned_auc:.4f}  (baseline: {baseline_auc:.4f}, gain: +{tuned_auc - baseline_auc:.4f})')
print(f'Tuned Accuracy: {tuned_acc*100:.1f}%  (baseline: {baseline_acc*100:.1f}%)')
print()
print(classification_report(y_test, y_pred, target_names=['Not Awarded', 'Awarded']))

## 7. Feature Importance — Tuned Model

In [ ]:
feat_imp = pd.Series(best_model.feature_importances_, index=FEATURES).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
feat_imp.plot(kind='barh', color='steelblue')
plt.title('Feature Importance — Tuned XGBoost', fontsize=14)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('=== Feature Ranking ===')
for feat, imp in feat_imp.iloc[::-1].items():
    print(f'  {feat:25s} {imp:.4f}')

## 8. Next Steps

- **Narrow with grid search** around the best `max_depth`, `learning_rate`, `n_estimators` values found above
- **Early stopping** — use `eval_set` with XGBoost to avoid overfitting on n_estimators  
- **SHAP values** — for interpretability beyond feature importance scores  
- **Plug in teammates' preprocessed data** — replace the data loading cell, FEATURES list should stay the same